# 1.15 參數不動，選字規則怎樣改變輸出？


同一題得到「貓、看、狗」分數 `[3,1,0]`。如果永遠取最高分，就選貓；如果按機率抽，另外兩字仍可能出現。這是使用同一模型的兩種選字規則，並沒有學到新參數。

總選最高者叫 **greedy**，即貪婪選取；`argmax` 給最高分的 ID。按分配隨機抽叫 **sampling**。還可以先用正數 T 除原分數，再做 softmax；T 叫**溫度**，只是控制分配尖銳程度的名稱。

T=0.5 使分數變 `[6,2,0]`，差距變大，最高者更佔優勢；T=2 使分數變 `[1.5,0.5,0]`，機率更平均。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

z = torch.tensor([3.0, 1.0, 0.0])
for temperature in [0.5, 1.0, 2.0]:
    probability = (z / temperature).softmax(dim=0)
    print(temperature, probability)
print("greedy的候選ID", z.argmax().item())

三行分配中，最高候選機率依序約 0.980、0.844、0.629。最後的 `argmax` 印 ID 0；用任何正溫度都不改分數排序，所以本例 greedy 仍選貓。溫度主要改抽樣時各字的機會。

用現在的參數算候選分數叫**推論**；生成反覆推論，再選字接回。改溫度改的是推論時使用的機率，更新參數才是訓練。高溫度可能增加多樣性，也可能增加不通順結果；低溫度也不保證正確。

練習只把第三個分數 0 改成 2。它的機率增加，三個溫度下最高者約為 0.867、0.665、0.506，greedy 仍選 ID 0。比較溫度時固定分數；比較模型訓練效果時，則固定溫度與選字方式，才能知道差別從哪裡來。

<details>
<summary>補充與重做</summary>

T 必須大於 0；「趨近 0」不等於直接設 0，除以 0 無法使用。若最高分並列，還要由實際取最大值規則處理。更換抽樣種子也可改文字，見 [1.14](https://birdhackor.github.io/tiny-perceptron-vlm/1.14.html)。

</details>
